# Experiment Lag - Paired same-day and previous-day weather

Compare same-day and previous-day weather on identical target dates. A separate model is fitted for each input design, so observed changes combine input-definition and fitted-model effects; they cannot be interpreted as the effect of changing only an explanation graph.

**Restart the kernel and Run All to compute, validate and display the complete experiment.** The setup cell computes results when inputs, source code or sampling settings have changed, and otherwise reuses validated results. Target-driven selection and eight-seed validation run from this notebook as well; no separate terminal command is needed. Complete recomputation takes longer than rendering cached results.

- **Population:** all 146 test target dates.
- **Difference direction:** modified minus baseline, in units of predicted daily rentals.
- **Computation:** all 128 coalitions of seven features; 500 causal imputations per coalition by default, and the complete training background for Marginal imputation.
- **Presentation:** population summaries, complete pointwise detail, target-driven local cases and independent-repeat evidence. Set `SHOW_FULL_TABLES` to expand long tables; use `SHOW_DIAGNOSTICS` for supplementary diagnostic views.

Shared computation and plotting code is in [experiment_support](experiment_support/). This notebook's configuration and execution order are the experiment entry point.


In [ ]:
from __future__ import annotations

import itertools
import sys
from pathlib import Path

# Locate this example from the repository root or an example subdirectory.
search_roots = [Path.cwd(), *Path.cwd().parents]
candidates = [
    candidate
    for parent in search_roots
    for candidate in (parent, parent / "examples" / "applications" / "causal_bike_sharing")
]
EXPERIMENT_ROOT = next(
    (
        path.resolve()
        for path in candidates
        if (path / "experiment_support" / "integrated_support.py").is_file()
    ),
    None,
)
if EXPERIMENT_ROOT is None:
    raise FileNotFoundError("Launch this notebook from the example directory or repository root.")
sys.path.insert(0, str(EXPERIMENT_ROOT / "experiment_support"))
from example_paths import configure_imports

configure_imports(EXPERIMENT_ROOT)

import numpy as np
import pandas as pd
from case_selection import DEFAULT_CONFIG, default_target_specs
from integrated_figures import (
    plot_first_order_distribution,
    plot_first_order_summary,
    plot_mean_interaction_matrices,
    plot_sample_heterogeneity,
)
from integrated_support import (
    FAMILIES,
    FEATURES,
    bootstrap_table,
    case_plots,
    change_network,
    csv,
    current_summary,
    design_table,
    ensure_results,
    figure,
    interaction_thresholds,
    load_experiment,
    paired_lag_variance,
    plot_intervals,
    plot_lag_diagnostics,
    rank_table,
    table,
)
from IPython.display import Markdown, display
from targeted_cases import ensure_targeted_cases, load_targeted_cases, render_targeted_cases

# The five notebooks share one validated complete computation.
RUN_MODE = "auto"
CAUSAL_SAMPLE_SIZE = 500  # Causal imputations per coalition; may be changed to 100 or 1000.
MARGINAL_SAMPLE_SIZE = "all"  # Use every training-background row.
SHOW_DIAGNOSTICS = False  # Extra matrices, legacy L1 cases and diagnostic views.
SHOW_FULL_TABLES = False  # Complete long tables remain available in collapsed form.
ARM = "lag"
PREFIX = "LAG"

ensure_results(
    EXPERIMENT_ROOT,
    mode=RUN_MODE,
    causal_sample_size=CAUSAL_SAMPLE_SIZE,
    marginal_sample_size=MARGINAL_SAMPLE_SIZE,
)
data = load_experiment(ARM)
phi_base, phi_modified = data["base_phi"], data["variant_phi"]
I_base, I_modified = data["base_interaction"], data["variant_interaction"]
delta_phi = phi_modified - phi_base
delta_I = I_modified - I_base
N_EXPLAIN = len(data["row_ids"])
assert N_EXPLAIN == 146
assert len(set(data["row_ids"])) == N_EXPLAIN
assert np.isfinite(delta_phi).all() and np.isfinite(delta_I).all()
print(f"Comparison {ARM}: all {N_EXPLAIN} test dates; result protocol and checks passed.")

_table_numbers = itertools.count(1)
_figure_numbers = itertools.count(1)


def show_table(frame, title, *, detail=False):
    table(
        frame,
        f"{PREFIX}-{next(_table_numbers)}",
        title,
        expanded=SHOW_FULL_TABLES if detail else True,
    )


def next_figure_number():
    return f"{PREFIX}-{next(_figure_numbers)}"


def show_case_evidence(case_data):
    """Legacy whole-explanation L1 cases; displayed only when diagnostics are enabled."""
    labels = {
        "typical": "Typical whole-explanation change",
        "strong": "Large whole-explanation change",
        "weak": "Small whole-explanation change",
        "high_rank_large_shift": "High ranks with a large whole-explanation shift",
    }
    for case in case_data["selected"].itertuples(index=False):
        row_id = int(case.row_id)
        display(
            Markdown(
                f"### {case_data['arm']} - {labels.get(case.case_rule, case.case_rule)}"
                f" - {case.date} (original row_id={row_id})\n\n"
                f"This legacy date is fixed before the independent imputation repeats. "
                f"Whole-explanation effect-score percentile: {case.effect_score_percentile:.1f}."
            )
        )
        change_network(
            case_data,
            next_figure_number(),
            row_id=row_id,
            case_rule=case.case_rule,
            top_k=3,
        )
        case_number = next_figure_number() if SHOW_DIAGNOSTICS else None
        case_plots(
            case_data,
            row_id,
            case.case_rule,
            case_number,
            next_figure_number(),
            show_discovery=SHOW_DIAGNOSTICS,
        )

## What the numbers represent

Nodes show **singleton terms in the order-2 k-SII decomposition**, and edges show separately allocated pair interactions. Singleton terms should not be equated with ordinary Shapley values. Edges in the input causal structure specify the imputation distribution; edges in the output interaction network display attribution results.

A random seed controls imputation draws, not the number of test dates. **Averages across dates** describe the population of explained cases; **repeated imputations for one fixed date** assess the computational stability of that case.


In [ ]:
_ = show_table(design_table(data), "Population, model and explanation settings")

Both designs exclude the original first day before splitting by the same target dates. Only temperature, humidity and wind speed use previous-day values in the Lag design; apparent temperature remains contemporaneous. Training targets and dates match, but input values, backgrounds and fitted models differ.


In [ ]:
quality = pd.DataFrame(
    [
        {"design": name, **value["metrics"]}
        for name, value in data["models"].items()
        if name in ("paired_same_day", "paired_lag")
    ]
)
_ = show_table(quality, "Predictive quality on the same target-date holdout")

## Changes across all test dates

**Mean signed change** retains direction. **Mean absolute change** takes the absolute difference for each date before averaging, so opposite changes do not cancel. **Mean magnitude change** compares absolute attribution values after and before modification. All three are paired by target date before aggregation.


In [ ]:
_ = show_table(
    data["item_summary"].query("order == 1"), "First-order k-SII means and paired changes"
)
number = next_figure_number()
figure(
    plot_first_order_summary(
        phi_base,
        phi_modified,
        feature_names=FEATURES,
        title_prefix=f"Figure {number}",
        variant_label=ARM,
    ),
    number,
)

The 95% intervals use 5,000 paired bootstrap resamples of test dates. They describe variation in sample composition conditional on the computed explanations. They do not cover imputation error, temporal dependence, or multiplicity across all configurations; imputation stability is checked separately for fixed local cases.


In [ ]:
signed_ci = bootstrap_table(delta_phi, FEATURES, n_boot=5000, seed=42)
_ = show_table(signed_ci, "Pointwise intervals for mean signed changes", detail=True)
if SHOW_DIAGNOSTICS:
    number = next_figure_number()
    figure(plot_intervals(signed_ci, f"Figure {number}: Mean signed changes"), number)

## Interactions and annotated mean networks

Nodes represent singleton terms and edges represent pair interactions. Baseline, modified and difference panels share a layout and the same displayed edges. To limit crowding, mean networks display the union of the three strongest edges in each panel; the complete table retains all 21 interactions. Mean-network strength is the **absolute value of a signed mean**, not the mean of per-date absolute values.

Targeted local networks below additionally retain every selected target pair, even when it is outside the strongest-edge subset.


In [ ]:
_ = show_table(
    data["item_summary"].query("order == 2"),
    "All 21 pairwise interactions and paired changes",
    detail=True,
)
change_network(data, next_figure_number(), top_k=3)
if SHOW_DIAGNOSTICS:
    number = next_figure_number()
    figure(
        plot_mean_interaction_matrices(
            I_base,
            I_modified,
            feature_names=FEATURES,
            title_prefix=f"Figure {number}",
            variant_label=ARM,
        ),
        number,
    )

## Paired date-level variability and interaction thresholds

The paired bootstrap uses the same resampled target-date indices for both designs. It compares attribution variability conditional on the computed explanations, rather than imputation-repeat uncertainty. Interaction counts use each arm's own relative threshold, so they describe within-arm concentration and are not counts exceeding one common absolute cutoff. These paired diagnostics remain visible by default.


In [ ]:
lag_variance = paired_lag_variance(
    phi_base,
    phi_modified,
    n_boot=5000,
    seed=42,
    feature_names=FEATURES,
)
thresholds = interaction_thresholds(I_base, I_modified)
_ = show_table(lag_variance["table"], "Paired date-bootstrap variance")
_ = show_table(thresholds, "Counts at arm-specific relative thresholds")
number = next_figure_number()
figure(
    plot_lag_diagnostics(lag_variance, thresholds, f"Figure {number}: Paired date comparison"),
    number,
)

## Complete results for every date and attribution term

The first table contains one row per date; the next two contain one row per date and feature, or per date and interaction. Collapsing a table changes presentation only and does not remove rows. Original row IDs connect population summaries, complete details and local networks.

The first-order L1 difference sums absolute changes across all seven singleton terms for a date; the mean L1 difference then averages across dates. Overview-scatter features are selected by their mean absolute change across the full population. Targeted cases below use declared individual features and pairs instead of that population-wide ranking.


In [ ]:
_ = show_table(data["points"], "Complete per-point comparison", detail=True)
feature_details = csv(f"tables/{ARM}_features.csv")
interaction_details = csv(f"tables/{ARM}_interactions.csv")
detail_columns = [
    "row_id",
    "date",
    "item",
    "baseline",
    "modified",
    "delta",
    "abs_delta",
    "magnitude_change",
    "sign_flip",
]
_ = show_table(feature_details[detail_columns], "Every day and all seven features", detail=True)
_ = show_table(
    interaction_details[detail_columns], "Every day and all 21 interactions", detail=True
)

In [ ]:
number = next_figure_number()
figure(
    plot_sample_heterogeneity(
        phi_base,
        phi_modified,
        feature_names=FEATURES,
        context_values=data["points"]["temp"].to_numpy(),
        context_label="Target-day temperature (C)",
        color_values=pd.to_datetime(data["points"]["date"]).dt.dayofyear.to_numpy(),
        color_label="Day of year",
        title_prefix=f"Figure {number}",
        variant_label=ARM,
    ),
    number,
)
if SHOW_DIAGNOSTICS:
    number = next_figure_number()
    figure(
        plot_first_order_distribution(
            phi_base,
            phi_modified,
            feature_names=FEATURES,
            title_prefix=f"Figure {number}",
            variant_label=ARM,
        ),
        number,
    )

## Rank agreement

For each date, first-order Kendall tau compares the seven signed singleton values before and after modification; the reported mean averages those per-date coefficients. Second-order tau compares the 21 pair interactions. Signed ranking and absolute-value ranking are distinct measures. Similar ranks do not imply small numerical changes.

The two Lag designs have different fitted models and input definitions. Their rank comparisons are descriptive and do not represent a fixed-model null-change test.


In [ ]:
_ = show_table(rank_table(data), "Rank agreement and conditional repeat reference")

## Target-driven cases and imputation repeats

Declare target singleton terms and feature pairs before selecting dates. Default targets include questions informed by earlier exploration; their origins are recorded in the target table. This is an exploratory analysis.

For each target, the display threshold is `max(min_absolute_change, baseline_scale_fraction * median(abs(baseline)))`, calculated across all test dates. Defaults are 25 rentals and 10% of the median baseline magnitude.

| Criterion | Eligibility | Selected representative |
|---|---|---|
| Largest change | Absolute target change meets its threshold | Largest absolute change |
| Material sign reversal | Opposite signs; both magnitudes meet the threshold | Largest absolute change |
| Stable ranks, large change | Same-order signed tau >= 0.90; change meets its threshold and 75th percentile | Largest absolute change |
| Typical change | All dates | Change closest to the signed median |
| Opposite direction | Material change opposite to the nonzero signed median | Largest absolute change |
| Opposing pair/singleton changes | A material pair change opposes a material change in a member singleton | Largest absolute pair change |

Each target/criterion retains one eligible date; exact ties use the lowest original row ID. Dates selected more than once are merged while retaining all reasons. Empty categories remain empty. The complete candidate table includes eligible and ineligible dates.

Selections are frozen before eight paired imputation repeats. Repeat seeds differ from the discovery seed; matching cached draws may be reused. Selected cases remain in the results regardless of their repeat stability. Repeat ranges measure computational variability, and the thresholds are display criteria rather than significance tests.

Edit `CASE_TARGETS` or set `CASE_SCAN_ALL=True` to scan all seven singleton terms and 21 pairs. `SHOW_DIAGNOSTICS` adds whole-explanation L1 examples.

Each case figure includes the date, seven model inputs, observed rentals and model predictions. Temperatures are in degrees Celsius, humidity in percent and wind speed on the dataset scale. Lag figures distinguish previous-day temperature, humidity and wind speed from same-day inputs. The exported readings JSON retains unrounded values.

In [ ]:
# Edit target specifications and thresholds before running the selection.
# These are exploratory display criteria, not significance cutoffs.
CASE_SELECTION_CONFIG = {
    **DEFAULT_CONFIG,
    "min_absolute_change": 25.0,
    "baseline_scale_fraction": 0.10,
    "rank_threshold": 0.90,
    "large_change_quantile": 0.75,
}
CASE_SCAN_ALL = False  # True includes all seven singleton terms and all 21 pairs.
CASE_TARGETS = {
    comparison: default_target_specs(comparison, scan_all=CASE_SCAN_ALL)
    for comparison in FAMILIES[ARM]
}
_ = show_table(
    pd.DataFrame(
        [{"setting": name, "value": value} for name, value in CASE_SELECTION_CONFIG.items()]
    ),
    "Editable target-selection thresholds",
)
for comparison, target_specs in CASE_TARGETS.items():
    _ = show_table(pd.DataFrame(target_specs), f"{comparison}: declared features and pairs")

targeted_bundle = ensure_targeted_cases(
    FAMILIES[ARM],
    root=EXPERIMENT_ROOT,
    mode=RUN_MODE,
    selection_config=CASE_SELECTION_CONFIG,
    targets=CASE_TARGETS,
)
focused = load_targeted_cases(ARM, root=EXPERIMENT_ROOT)

In [ ]:
render_targeted_cases(
    focused,
    prefix=f"{PREFIX}F",
    root=EXPERIMENT_ROOT,
    expanded_tables=SHOW_FULL_TABLES,
)

if SHOW_DIAGNOSTICS:
    _ = show_table(data["selection_rules"], "Legacy L1 case rules", detail=True)
    _ = show_table(data["selected"], "Legacy L1 selected dates", detail=True)
    _ = show_table(data["validation_summary"], "Legacy L1 case repeat stability")
    _ = show_table(data["validation"], "All legacy L1 paired repeats", detail=True)
    show_case_evidence(data)

## Inspect any target date

Set `INSPECT_ROW_ID` at the start of the next cell and run that cell to inspect all seven singleton terms, all 21 pair interactions and the network for that date. Set `INSPECT_ARM` to inspect another configuration included in the experiment.

An arbitrary date is a descriptive inspection. Independent-repeat evidence is available for the fixed cases selected and validated in the target-driven section above.


In [ ]:
INSPECT_ROW_ID = None  # Enter an original row ID, then run this cell.
INSPECT_ARM = None  # None uses this notebook's main comparison; e.g. "b3" selects another.

if INSPECT_ROW_ID is None:
    print("Set INSPECT_ROW_ID and rerun this cell to inspect a date's complete attribution.")
else:
    inspected_arm = INSPECT_ARM or ARM
    inspected = data if inspected_arm == ARM else load_experiment(inspected_arm)
    if isinstance(INSPECT_ROW_ID, bool) or not float(INSPECT_ROW_ID).is_integer():
        raise ValueError("INSPECT_ROW_ID must be an integer original row ID.")
    row_id = int(INSPECT_ROW_ID)
    if row_id not in set(inspected["row_ids"]):
        raise ValueError(
            "This row ID is outside the comparison's test dates; use the complete point table."
        )
    for kind in ("features", "interactions"):
        detail = csv(f"tables/{inspected_arm}_{kind}.csv")
        _ = show_table(
            detail.loc[detail.row_id == row_id, detail_columns],
            f"{inspected_arm}: row {row_id}, {kind}",
        )
    change_network(inspected, next_figure_number(), row_id=row_id, top_k=3)

## Interpreting the evidence

Population means describe prevalence and average direction. Per-date and per-term results identify where changes occur. Repeated imputations of the same fixed case assess dependence on a particular random draw. An extreme case, a large absolute difference or a high rank correlation alone cannot establish that a causal structure is correct.

The current protocol, model information and complete numerical results are stored in [results](results/). Target-driven records include the editable selection specification, every candidate, the retained date/target/reason and independent-repeat results. Notebook figures and tables are exported through the shared rendering helpers. The setup cell validates result provenance before any output is used.


In [ ]:
_summary = current_summary(data, family=FAMILIES[ARM], variance=lag_variance)
print(
    "All results shown above use validated computations. Enable SHOW_DIAGNOSTICS for supplementary views."
)